In [1]:
import json


class Data:
    """
    Data class for game/corpus
    @param question: question of the game/corpus
    @param answer: answer of the game/corpus
    @param difficulty: difficulty of the game/corpus, from 1 to 10
    """

    def __init__(self, question: str, answer: str, difficulty: int = 1, metadata: dict = None, **kwargs):
        self.question = question
        self.answer = answer
        self.difficulty = difficulty
        self.metadata = metadata
        self.gpt_response = ""

    def to_json(self):
        return {
            "question": self.question,
            "answer": self.answer,
            "difficulty": self.difficulty,
            "metadata": self.metadata,
            "gpt_response": self.gpt_response,
        }

    def to_json_str(self):
        return json.dumps(self.to_json(), ensure_ascii=False)

    @classmethod
    def from_json_str(cls, json_str):
        json_data = json.loads(json_str)
        return cls(**json_data)

    @classmethod
    def from_json_dict(cls, json_dict):
        instance = cls(**json_dict)
        if "gpt_response" in json_dict:
            instance.gpt_response = json_dict["gpt_response"]
        return instance

    @classmethod
    def from_jsonl_file(cls, file_path):
        data_list = []
        with open(file_path, "r") as f:
            for line in f:
                json_data = json.loads(line)
                instance = cls(**json_data)
                if "gpt_response" in json_data:
                    instance.gpt_response = json_data["gpt_response"]
                data_list.append(instance)
        return data_list


In [2]:
import random

def generate_pendolf_dataset(num_samples: int):
    items = ["кристаллы", "хвосты крыс", "зелья маны"]
    dataset = []

    for _ in range(num_samples):
        item = random.choice(items)
        # 50/50: либо игрок молодец, либо пытается обмануть
        is_success = random.choice([True, False]) 

        if is_success:
            # Сценарий 1: Честная сдача
            # Метаданные (стейт БД) понадобятся позже для симуляции среды
            meta = {"inventory": {item: 3}, "quest_active": True}
            q = f"Эй, Пендольф, я принес 3 {item}!"
            
            # Идеальная траектория, которую должна выучить Qwen
            a = (
                f"Мысль: Нужно проверить карманы этого умника.\n"
                f"Action: check_inventory('{item}')\n"
                f"Observation: 3\n"
                f"Action: take_item('{item}', 3)\n"
                f"Пендольф: А, не соврал! Давай сюда {item}, нечего им пылиться. Держи монеты, заслужил."
            )
        else:
            # Сценарий 2: Попытка обмана
            meta = {"inventory": {item: 0}, "quest_active": True}
            q = f"Я собрал все {item}, старик, давай награду."
            
            a = (
                f"Мысль: Проверим-ка его инвентарь, звучит подозрительно.\n"
                f"Action: check_inventory('{item}')\n"
                f"Observation: 0\n"
                f"Пендольф: Ты кого пытаешься надуть?! У тебя ни одного {item} в сумке нет! Пшел вон, пока я тебя в жабу не превратил!"
            )

        # Упаковываем в твой класс Data
        dataset.append(Data(question=q, answer=a, difficulty=1, metadata=meta))

    return dataset

In [5]:
# 1. Генерируем 5 примеров
pendolf_data = generate_pendolf_dataset(5)

# 2. Смотрим, что получилось (выводим первый элемент)
print("--- Пример сгенерированного сэмпла ---")
print(pendolf_data[0].to_json_str())

--- Пример сгенерированного сэмпла ---
{"question": "Я собрал все хвосты крыс, старик, давай награду.", "answer": "Мысль: Проверим-ка его инвентарь, звучит подозрительно.\nAction: check_inventory('хвосты крыс')\nObservation: 0\nПендольф: Ты кого пытаешься надуть?! У тебя ни одного хвосты крыс в сумке нет! Пшел вон, пока я тебя в жабу не превратил!", "difficulty": 1, "metadata": {"inventory": {"хвосты крыс": 0}, "quest_active": true}, "gpt_response": ""}
